In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df=pd.read_csv('C:/a/run/hour.csv')
df

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,17375,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,11,108,119
17375,17376,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,8,81,89
17376,17377,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,7,83,90
17377,17378,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,13,48,61


In [95]:
features=df.drop(['cnt','dteday','instant'], axis=1)
label=df['cnt']

In [96]:
features

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,11,108
17375,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,8,81
17376,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,7,83
17377,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,13,48


In [97]:
label

0         16
1         40
2         32
3         13
4          1
        ... 
17374    119
17375     89
17376     90
17377     61
17378     49
Name: cnt, Length: 17379, dtype: int64

In [102]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test=train_test_split(features,label,test_size=0.2, random_state=1)

scalar=StandardScaler()
x_train_scaled=scalar.fit_transform(X_train)
y_test_scaled=scalar.transform(X_test)

In [103]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [104]:
model_configs=[
    {'name':'DecisionTree', 'estimator':DecisionTreeRegressor(random_state=1), 'param':{'max_depth':[5,10,15]}},
    {'name':'GradientBoosting', 'estimator':GradientBoostingRegressor(random_state=1), 'param':{'n_estimators':[100,200], 'max_depth':[5,8],'learning_rate':[0.05,0.1]}}
]

In [106]:
result={}

for config in model_configs:
    model_name=config['name']
    print(f"{model_name} loading...")
    grid_search=GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['param'],
        cv=5,
        scoring='neg_mean_squared_error',
        refit=True,
        n_jobs=1,
    )

    grid_search.fit(x_train_scaled, y_train)

    best_model=grid_search.best_estimator_
    y_pred=best_model.predict(y_test_scaled)

    result[model_name]={
        'best_params':grid_search.best_params_,
        'best_neg_mse':grid_search.best_score_,
        'test_r2':r2_score(y_test, y_pred)
    }

results_df=pd.DataFrame(result).T.sort_values(by='test_r2', ascending=False)
print(results_df)


DecisionTree loading...
GradientBoosting loading...
                                                        best_params best_neg_mse   test_r2
GradientBoosting  {'learning_rate': 0.1, 'max_depth': 5, 'n_esti...    -7.988604  0.999812
DecisionTree                                      {'max_depth': 15}   -33.681252  0.999234


In [ ]:
# DecisionTreeRegressor는 한 번에 데이터를 읽어 특정 조건에 맞게 분할 후 평균값을 생성해 예측하지만
# GradientBoostingRegressor는 순차적으로 데이터를 읽어 오류를 해결하면서 더 정확한 예측을 가능케 합니다.
# 그래서 GradientBoostingRegressor 모델이 일반적으로 더 높은 성능을 가집니다.